In [6]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.11.2
numpy 2.5.3
pandas 3.0.5
sklearn 1.9.1
torch 2.14.0
cpu


# 加载数据

In [7]:
from torchvision import datasets
from torchvision.transforms import ToTensor

train_ds = datasets.FashionMNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=ToTensor()
)

test_ds = datasets.FashionMNIST(
    root='./data', 
    train=False, 
    download=True,
    transform=ToTensor()
)

# torchvision 数据集里没有提供训练集和验证集的划分
# 当然也可以用 torch.utils.data.Dataset 实现人为划分
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False)

In [8]:
from torchvision.transforms import Normalize

# 遍历 train_ds 得到每张图片，计算每个通道的均值和方差
def cal_mean_std(ds):
    mean = 0.
    std = 0.
    for img, _ in ds:
        mean += img.mean(dim=(1, 2))
        std += img.std(dim=(1, 2))
    mean /= len(ds)
    std /= len(ds)
    return mean, std

print(cal_mean_std(train_ds))
"""
nn.Sequential 是 PyTorch中用来按顺序执行多个模块的容器
这里 0.2860表示μ均值，0.3205表示σ标准差
为什么是列表 [0.2860]：因为图片通常有多个通道（每个通道有一个均值、一个标准差）
"""
transforms = nn.Sequential(
    Normalize([0.2860], [0.3205])               # 这里的均值和标准差是通过train_ds计算得到的
)

(tensor([0.2860]), tensor([0.3205]))


# 定义模型

这里我们没有用`nn.Linear`的默认初始化，而是采用了xavier均匀分布去初始化全连接层的权重

xavier初始化出自论文 《Understanding the difficulty of training deep feedforward neural networks》，适用于使用`tanh`和`sigmoid`激活函数的方法。当然，我们这里的模型采用的是`relu`激活函数，采用He初始化（何凯明初始化）会更加合适。感兴趣的同学可以自己动手修改并比对效果。

|神经网络层数|初始化方式|early stop at epoch| val_loss | vla_acc|
|-|-|-|-|-|
|20|默认|
|20|xavier_uniform|
|20|he_uniform|
|...|

He初始化出自论文 《Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification》

In [9]:
class NeuralNetwork(nn.Module):
    def __init__(self, layers_num=2):
        super().__init__()
        """
        在__init__里定义的层，其参数会被nn.Module自动注册，从而能被优化器找到、被GPU移动、被保存/加载；
        而forward里临时创建的层，参数是“野”的，训练根本进行不下去
        如果层是在forward中临时创建的：
            def forward(self, x):
                fc = nn.Linear(10, 5)   # 每次都新建，参数不在 model.parameters() 里
                return fc(x)
                
        深度学习本质（同一组参数，在成千上万次前向传播中反复使用，通过梯度不断更新）：
            层定义在__init__：参数只创建一次，之后每次forward都复用同一组权重
            层定义在forward：每次调用都重新创建，参数是新的随机值，上一次学到的梯度无处累积
        """
        self.transforms = transforms
        self.flatten = nn.Flatten()
        
        # linear_relu_stack：非官方术语，作者自己起的变量名，意思是有Linear（线性层）和ReLU（激活函数）堆叠（stack）而成的序列
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 100),
            nn.ReLU(),
        )
        
        for i in range(1, layers_num):
            # add_module(name, module)：往Sequential里追加模块，并给名字
            """
            维度不变，不代表什么都没做。它是一个同维度的线性变换：
                特征重组/重新组合：把上层100个特征，通过权重矩阵混合、加权求和，生成100个新特征。每个特征都是原100个特征的线性组合
                增加非线性表达能力：虽然这一层本身是线性的，但它后面接ReLU，所以整体是 线性变换+非线性激活。多叠几层，网络就能拟合更复杂的函数
                加深网络：增加深度，提升表达能力（万能逼近定理：宽度和深度都能增强表达能力，深度往往更高效）
            所以它的意义不是“改变维度”，而是在保持维度的前提下，对特征做更深层的变换
            
            为什么保持100维不变：
                结构统一、好写循环：中间隐藏层都用同样维度，方便用for循环批量添加
                参数量可控：每层参数量固定为 100*100，不会突然爆炸
                特征容量稳定：不增不减，让信息在同一“带宽”里反复加工
                
            Linear(784, 100)：降维，把784维输入压到100维，提取主要特征
            Linear(100, 100)：同维变换，在100维空间里反复加工特征
            Linear(100, 10)：降维到输出，映射到10个类别得分
            """
            self.linear_relu_stack.add_module(f'Linear_{i}', nn.Linear(100, 100))
            self.linear_relu_stack.add_module(f'relu_{i}', nn.ReLU())
            
        self.linear_relu_stack.add_module('Output Layer', nn.Linear(100, 10))
        
        self.init_weights()
        
    def init_weights(self):
        for m in self.modules():
            # 如果是全连接层，进行参数和偏置值初始化
            if isinstance(m, nn.Linear):
                # uniform 均匀分布  normalization 正态分布
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
                
    def forward(self, x):
        x = self.transforms(x)
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
total = 0
"""
遍历一个20层神经网络的所有参数，同时拿到每个参数的序号、名字和张量值
NeuralNetwork(20)：创建一个神经网络，20表示层数（比如20个隐藏层）
.named_parameters()：这是nn.Module方法，返回一个迭代器，每次产出一个（名字，参数张量）元组
    名字（key）：参数在模型中的路径，如linear_relu_stack.0.weight
    参数（value）：对应nn.Parameter张量
enumerate():
    给迭代器加上序号，从0开始
为什么这样解包：
    enumerate产出的是（序号，元素），而元素本身是（key，value）。所以结构是：(idx, (key, value))
为什么idx直接从0，1直接跳到 3 了：
    因为编号是nn.Sequential里模块的“位置索引”，而位置2被ReLU占了——ReLU没有可学习参数，所以不出现在named_parameters()里，但它的位置仍然被计数
    
这里在循环的过程中，不是只循环线性层（全连接层），而是你的NeuralNetwork(20)里，除了Linear之外，其他层都没有可学习参数（比如ReLU），而named_parameters()只返回有参数的层：
    它遍历模型里所有可学习的参数（nn.Parameter），包括：
        Linear的weight和bias
        BatchNorm的weight和bias
        LSTM、Conv等的参数
        任何requires_grad=True的Parameter
    不返回没有参数的层，如：
        nn.ReLU()：没有参数
        nn.Flatten()：没有参数
        nn.MaxPool2d()：没有参数
        nn.Dropout()：没有参数
    如果模型里只有Linear有参数，那么遍历结果自然全是Linear的参数
"""
for idx, (key, value) in enumerate(NeuralNetwork(20).named_parameters()):
    """
    idx // 2：向下取整
    02:
        0：用0填充空位
        2: 总宽度为2
    \t : 制表符（tab），在输出里制造一个对齐的空白间隔，让“层名”和“参数量”分列显示
    np.prod(...)：把形状里所有维度相乘，得到元素总数，也就是参数量
        np.prod 是 NumPy 的函数，对torch.Size（本质是元组）也能用
        prod: product，乘积
    """
    print(f'Linear_{idx // 2:>02}\tparameters num: {np.prod(value.shape)}')
    # 累加每一层参数量，最终得到整个模型的总参数量
    total += np.prod(value.shape)
    
total

Linear_00	parameters num: 78400
Linear_00	parameters num: 100
Linear_01	parameters num: 10000
Linear_01	parameters num: 100
Linear_02	parameters num: 10000
Linear_02	parameters num: 100
Linear_03	parameters num: 10000
Linear_03	parameters num: 100
Linear_04	parameters num: 10000
Linear_04	parameters num: 100
Linear_05	parameters num: 10000
Linear_05	parameters num: 100
Linear_06	parameters num: 10000
Linear_06	parameters num: 100
Linear_07	parameters num: 10000
Linear_07	parameters num: 100
Linear_08	parameters num: 10000
Linear_08	parameters num: 100
Linear_09	parameters num: 10000
Linear_09	parameters num: 100
Linear_10	parameters num: 10000
Linear_10	parameters num: 100
Linear_11	parameters num: 10000
Linear_11	parameters num: 100
Linear_12	parameters num: 10000
Linear_12	parameters num: 100
Linear_13	parameters num: 10000
Linear_13	parameters num: 100
Linear_14	parameters num: 10000
Linear_14	parameters num: 100
Linear_15	parameters num: 10000
Linear_15	parameters num: 100
Linear_1

np.int64(271410)

In [10]:
"""
创建一个形状为3行5列的张量
empty表示只分配内存，不初始化数值——里面的值是内存里“垃圾值”，可能是任意数（不是0，也不是随机数）
用empty是因为后面马上用 nn.inti.eye_覆盖它，所以没必要先花时间填值
"""
w = torch.empty(3, 5)
"""
eye是单位矩阵的英文俗称，代表"Identity matrix"(单位矩阵)；
末尾的_表示原地操作（in-place）
    单位矩阵常用I，eye谐音
    单位矩阵“1”分布在对角线上，看起来像一只眼睛
nn.init 是PyTorch的初始化模块（torch.nn.init）
eye_是其中一个函数，把张量原地（in-place）初始化为单位矩阵
    当初始化矩阵非 n阶 方阵时，沿左上角对角线赋值为1
末尾的下划线_表示原地操作，直接修改w本身，而不是返回一个新张量
"""
nn.init.eye_(w)

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.]])

# 训练

In [11]:
from sklearn.metrics import accuracy_score

@torch.no_grad()
def evaluating(model, dataloader, loss_fct):
    loss_list = []
    pred_list = []
    label_list = []
    for datas, labels in dataloader:
        datas = datas.to(device)
        labels = labels.to(device)
        
        logits = model(datas)
        loss = loss_fct(logits, labels)
        loss_list.append(loss.item())
        """
        torch.max：既返回最大值本身（values），也返回最大值所在的索引（indices）
        torch.argmax：只返回最大值所在的索引（indices）
        """
        preds = logits.argmax(axis=-1)
        """
        在这段代码里，他们被展平成一维列表（extend而不是append），所以accuracy_score收到的是一维数组，而不是二维的“每个batch一个子列表”
        """
        pred_list.extend(preds.cpu().numpy().tolist())
        label_list.extend(labels.cpu().numpy().tolist())
        
    acc = accuracy_score(label_list, pred_list)
    return np.mean(loss_list), acc


In [ ]:
"""
TensorBoard 原本是 TensorFlow的可视化工具，用来：
    查看loss、accuracy曲线
    查看计算图（网络结构）
    查看权重、梯度的分布直方图
    查看图像、音频、文本等
PyTorch通过torch.utils.tensorboard提供了兼容接口，不需要安装TensorFlow，只要装tensorboard包即可
from torch.utils.tensorboard import SummaryWriter：
    从PyTorch中导入TensorBoard的写入器SummaryWriter，用于记录和可视化训练过程
    torch.utils.tensorboard     PyTorch集成的TensorBoard工具模块
    SummaryWriter               核心类，负责把数据写入日志文件
    from ... import ...         只导入SummaryWriter这一个类
"""
from torch.utils.tensorboard import SummaryWriter
"""
TensorBoardCallback：这是一个自定义的回调（Callback）类，用于把训练过程中的数据写入TensorBoard日志
TensorBoardCallback封装了SummaryWriter，通常配合训练循环使用，让“记录日志”这件事从训练代码里解耦出来，变成可复用、可插拔的组件
"""
class TensorBoardCallback:
    def __init__(self, log_dir, flush_secs=10):
        """
        :param log_dir: 日志保存目录（TensorBoard读取的路径）
        :param flush_secs: 多久把缓冲区数据刷写到磁盘一次，默认10秒
        创建SummaryWriter实例并保存为属性：
            log_dir：指定日志目录，如runs/exp1
            flush_secs：异步写入的刷新间隔。设太小会频繁写盘、略慢；设太大数据可能延迟落盘。默认10秒是平衡值
        SummaryWriter默认不是每写一条就立刻落盘，而是先放内存缓冲，定期（每flush_secs秒）批量写入磁盘
        训练结束或调用writer.close()时，会强制刷新剩余数据
        """
        self.writer = SummaryWriter(log_dir=log_dir, flush_secs=flush_secs)
    """
    把模型的计算图（网络结构）写入TensorBoard，可以在GRAPHS面板查看
        model                       要绘制的模型（nn.Module实例）
        input_shape                 输入张量的形状，如(1, 1, 28, 28)
        torch.randn(input_shape)    生成一个符合该形状的随机输入，用于追踪计算图
    关键点：
        add_graph 需要一次真实的前向传播来追踪数据流，所以必须提供输入
        输入形状要和模型实际接收一致，否则追踪失败或图不对
        input_shape通常要带batch维度，比如(1, 1, 28, 28)而不是(1, 28, 28)
    使用示例：
        tb.draw_model(model, input_shape=(1, 1, 28, 28))
        之后运行 tensorboard --logdir runs，在GRAPHS标签页就能看到网络结构
    两种写法关系：
        tensorboard --logdir=runs   用等号
        tensorboard --logdir runs   用空格
        完全等价，TensorBoard都认。这是argparse通用规则
            --参数=值 和 --参数 值 效果一样
            值里不含空格时，两种写法都行
            值里含空格时，必须加引号：--logdir="my runs/exp 1"
    """
    def draw_model(self, model, input_shape):
        self.writer.add_graph(model, input_to_model=torch.randn(input_shape))
    """
    同图记录训练/验证损失：
        作用：把训练损失和验证损失画在同一张图里，方便对比
        main_tag                    主标签，图的名字（分组用/）
        tag_scalar_dict             字典，每个key是一条曲线，value是对应数值
        global_step                 横坐标（通常是epoch或step）
    效果：
        training/loss 下会有两条曲线：loss和val_loss
        两条线共享同一坐标轴，一眼看出训练集和验证集是否过拟合
    """
    def add_loss_scalars(self, step, loss, val_loss):
        self.writer.add_scalars(
            main_tag="training/loss",
            tag_scalar_dict={"loss": loss, "val_loss": val_loss},
            global_step=step,
        )
    """
    记录学习率变化：
        作用：记录学习率随step的变化曲线，常用于检查学习率调度器（scheduler）是否按预期工作
    为什么重要：
        学习率是训练的关机超参数，画出来能直观看到warmup、衰减、阶梯下降等策略
        如果loss异常，看一眼lr曲线往往能定位问题（比如lr突然爆炸或降到0）
    """
    def add_lr_scalars(self, step, learning_rate):
        self.writer.add_scalars(
            main_tag="training/learning_rate",
            tag_scalar_dict={"learning_rate": learning_rate},
            global_step=step,
        )
    """
    __init__        创建实例时自动调用一次             初始化对象，设置属性
    __call__        把实例当函数调用时触发             让实例可被调用，执行某段逻辑
    class Foo:
        def __init__(self, x):
            self.x = x
    
        def __call__(self, y):
            print("call 被调用")
            return self.x + y
    
    f = Foo(10)      # 触发 __init__
    result = f(5)    # 触发 __call__，输出：call 被调用
    print(result)    # 15
    
    这是把TensorBoardCallback示例变成可调用的，用一个统一的入口tb.(step, **kwargs)根据传入的关键字自动记录对应的指标
    整体作用：tb(step, **kwargs)
        tb(step, loss=..., val_loss=..., acc=..., val_acc=..., lr=...)
        传什么就记什么，没传就跳过，调用方不用关心哪些指标该记、怎么记
    """
    def __call__(self, step, **kwargs):
        """
        __call__：让实例能像函数一样被调用，即tb(...)
        :param step: 必填，作为横坐标（通常是epoch或global step）
        :param kwargs: 收集所有关键字参数，比如 loss=0.5, acc=0.9
        kwargs.pop("loss", None)：从kwargs里取出loss，取不到就返回None
        用pop而不是get：取出的同时从字典里删掉，避免后面重复处理或残留
        只有当loss和val_loss都存在时才记录（因为add_loss_scalars要同时画两条线）
        """
        loss = kwargs.pop("loss", None)
        val_loss = kwargs.pop("val_loss", None)
        if loss is not None and val_loss is not None:
            self.add_loss_scalars(step, loss, val_loss)
            
        # 与上同理
        acc = kwargs.pop("acc", None)
        val_acc = kwargs.pop("val_acc", None)
        if acc is not None and val_acc is not None:
            self.add_acc_scalars(step, acc, val_acc)
        """
        学习率确实是超参数，但它可以在训练过程中被“调度器(scheduler)”动态改变，所以画出来就是一条随step/epoch变化的曲线
            超参数：训练前人为设定、不由梯度更新的参数，如学习率、batch size、层数
            固定超参数：整个训练过程保持不变的超参数
            动态超参数：训练过程中按规则改变的超参数，学习率最典型
        “超参数”和“固定不变”不是一回事。学习率是超参数，但它可以被调度器改写，所以能有曲线
        """
        # 学习率通常只有一个值（没有“验证学习率”），所以只判断lr是否存在；存在就记录
        learning_rate = kwargs.pop("lr", None)
        if learning_rate is not None:
            self.add_lr_scalars(step, learning_rate)
            

In [ ]:
class SaveCheckpointsCallback:
    def __init__(self, log_dir, save_step=5000, save_best_only=True):
        self.save_dir = log_dir
        self.save_step = save_step
        self.save_best_only = save_best_only
        self.best_metrics = -1
        
        if not os.path.exists(self.save_dir):
            os.mkdir(self.save_dir)
            
    def __call__(self, step, state_dict, metric=None):
        if step % self.save_step > 0:
            return
        
        if self.save_best_only:
            assert metric is not None
            if metric >= self.best_metrics:
                torch.save(state_dict, os.path.join(self.save_dir, "best.ckpt"))
                self.best_metrics = metric
        else:
            torch.save(state_dict, os.path.join(self.save_dir, f"{step}.ckpt"))

In [ ]:
class EarlyStopCallback:
    def __init__(self, patience=5, min_delta=0.01):
        self.patience = patience
        self.min_delta = min_delta
        self.best_metric = -1
        self.counter = 0
        
    def __call__(self, metric):
        if metric >= self.best_metric + self.min_delta:
            self.best_metric = metric
            self.counter = 0
        else:
            self.counter += 1
            
    @property
    def early_stop(self):
        return self.counter >= self.patience

In [ ]:
"""
这是Python语法明确允许的，叫trailing comma（尾随逗号）。它不只用于函数参数，还适用于：
    函数调用：foo(a, b, c,)
    列表/元组/字典：[1, 2, 3]、(1, 2,)、{"a", 1,}
    导入语句：from module import a, b, c,
Python官方语法（PEP 8也推荐）允许并鼓励在多行结构中保留尾随逗号
"""
def training(
        model, 
        train_loader,
        val_loader,
        epoch,
        loss_fct,
        optimizer,
        tensorboard_callback=None,
        save_ckpt_callback=None,
        early_stop_callback=None,
        eval_step=500,
):
    record_dict = {"train": [],
                   "val": []}
    global_step = 0
    model.train()
    with tqdm(total=epoch * len(train_loader)) as pbar:
        for epoch_id in range(epoch):
            for datas, labels in train_loader:
                datas = datas.to(device)
                labels = labels.to(device)
                # 梯度清零
                optimizer.zero_grad()
                # 前向传播
                logits = model(datas)
                # 计算损失，这里损失是一个张量（因为logits本身就是一个张量）
                loss = loss_fct(logits, labels)
                # 因为是张量，所以loss才能反向传播，梯度才能回传
                loss.backward()
                # 调整优化器，包括学习率的变动
                optimizer.step()
                # 预测标签
                preds = logits.argmax(axis=-1)
                # 准确率
                acc = accuracy_score(labels.cpu().numpy(), preds.cpu().numpy())
                # 只要损失值，无需损失张量
                loss = loss.cpu().item()
                
                record_dict["train"].append({
                    "loss": loss,
                    "acc": acc,
                    "step": global_step
                })
                # 一个epoch有多少个batch，global_step就增加多少次
                # eval_step=500表示每500个batch做一次评估，不是每500个epoch
                if global_step % eval_step == 0:
                    # evaluating
                    model.eval()
                    val_loss, val_acc = evaluating(model, val_loader, loss_fct)
                    record_dict["val"].append({
                        "loss": val_loss,
                        "acc": val_acc,
                        "step": global_step
                    })
                    model.train()
                    # 1. 使用 tensorboard 可视化
                    if tensorboard_callback is not None:
                        tensorboard_callback(
                            global_step,
                            loss=loss,
                            val_loss=val_loss,
                            acc=acc,
                            val_acc=val_acc,
                            lr=optimizer.param_groups[0]["lr"],
                        )
                    # 2. 保存模型权重 save model checkpoint
                    if save_ckpt_callback is not None:
                        save_ckpt_callback(global_step, model.state_dict(), metric=val_acc)
                    # 3. 早停 Early Stop
                    if early_stop_callback is not None:
                        early_stop_callback(val_acc)
                        if early_stop_callback.early_stop:
                            print(f"Early stop at epoch {epoch_id} / global_step {global_step}")
                            return record_dict
                        
                # update step（global_step本身不是以epoch为单位递增的，它通常以batch（或optimizer step）为单位递增）
                global_step += 1
                pbar.update(1)
                pbar.set_postfix({"epoch": epoch_id})
                
    return record_dict

epoch = 100

model = NeuralNetwork(layers_num=10)